In [ ]:
# dementia_qwen3_predict_cot.py
import os
import re
import sys
from pathlib import Path
import torch
import pandas as pd
from typing import List, Dict, Optional
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

# ----------------- CONFIG -----------------
DATA_DIR = r"E:\Thesis\cookie_control_dementia"
MODEL_NAME = "Qwen/Qwen3-4B"  # Using Qwen3-4B from HuggingFace
HF_TOKEN = os.environ.get("HF_TOKEN")
OUTPUT_CSV = r"E:\Thesis\qwen3_4B_cot_binaryclass.csv"

GEN_MAX_NEW_TOKENS = 1024  # Increased for COT reasoning
GEN_TEMPERATURE = 0.3
GEN_TOP_P = 0.9

# ----------------- HF SANITY CHECK -----------------
def hf_sanity_check(model_name: str, token: Optional[str] = None) -> bool:
    """Verify model is accessible on HuggingFace Hub."""
    api = HfApi()
    try:
        info = api.model_info(model_name, token=token)
        print(f"✅ HF CHECK OK: {info.modelId} (private={info.private})")
        return True
    except Exception as e:
        print(f"❌ HF CHECK ERROR: {type(e).__name__}: {e}", file=sys.stderr)
        return False

# ----------------- TOKENIZER / MODEL LOADERS -----------------
def load_tokenizer(model_name: str, hf_token: Optional[str] = None):
    """Load tokenizer for the specified model."""
    try:
        tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=hf_token)
        print(f"✅ Tokenizer loaded: {model_name}")
    except Exception as e:
        print(f"❌ [Tokenizer] Failed to load: {e}", file=sys.stderr)
        raise
    
    if not getattr(tok, "chat_template", None):
        tok.chat_template = (
            "{% for m in messages %}"
            "{{ m['role'] }}: {{ m['content'] }}\n"
            "{% endfor %}"
            "assistant:"
        )
    
    return tok

def load_model(model_name: str, hf_token: Optional[str] = None, 
               torch_dtype=torch.float16):
    """Load model from HuggingFace Hub."""
    print(f"[Model] Loading: {model_name}")
    
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, 
            trust_remote_code=True, 
            torch_dtype=torch_dtype, 
            device_map="auto",
            token=hf_token
        )
        print(f"✅ Model loaded successfully: {model_name}")
        return model
    except Exception as e:
        print(f"❌ Failed to load {model_name}: {type(e).__name__}: {e}", file=sys.stderr)
        raise

# ----------------- PROMPT / PARSING -----------------
def extract_patient_info(filename: str):
    match = re.match(r'(\d{3})', filename)
    return match.group(1) if match else None

def create_cot_prompt(filename: str, transcript: str) -> str:
    """Create a Chain-of-Thought prompt for dementia diagnosis."""
    return f"""You are an expert neuropsychologist specializing in dementia assessment. Analyze the following Cookie Theft picture description transcript.

**File:** {filename}

**Transcript:**
{transcript}

**Task:**
Analyze this transcript step by step to determine if this person shows signs of dementia.

**Think through the following aspects:**

1. **Language Fluency**: Is the speech fluent or hesitant? Are there many pauses, fillers, or restarts?

2. **Vocabulary & Word Finding**: Does the speaker use specific words or vague terms? Are there word-finding difficulties or circumlocutions?

3. **Content & Completeness**: Does the speaker describe the main elements of the Cookie Theft picture (mother, children, stool, cookies, sink overflow, window)?

4. **Grammar & Syntax**: Are sentences well-formed or fragmented? Is there evidence of grammatical errors?

5. **Coherence & Organization**: Is the description organized and logical, or scattered and disjointed?

6. **Repetitions**: Are there unnecessary repetitions of words, phrases, or ideas?

**Step-by-step reasoning:**
[Provide your detailed analysis for each aspect above]

**Final Assessment:**
Based on your step-by-step analysis above, provide your prediction:

Label: [ProbableAD/Control]
MMSE: [0-30]
Confidence: [Low/Medium/High]"""

def parse_llm_response(response: str) -> Dict:
    """Parse LLM response to extract label, MMSE, and confidence."""
    mmse_match = re.search(r'MMSE:\s*(\d+)', response)
    conf_match = re.search(r'Confidence:\s*(\w+)', response)
    
    # More flexible matching for ProbableAD or Control
    label = 'Unknown'
    response_lower = response.lower()
    
    # Look for final assessment section first
    final_section = response_lower.split('final assessment')[-1] if 'final assessment' in response_lower else response_lower
    
    # Check for ProbableAD variations
    if re.search(r'label:\s*probable\s*ad', final_section) or re.search(r'label:\s*probablead', final_section):
        label = 'ProbableAD'
    elif re.search(r'label:\s*control', final_section):
        label = 'Control'
    # Fallback: check entire response
    elif re.search(r'probable\s*ad', response_lower) or 'probablead' in response_lower:
        label = 'ProbableAD'
    elif re.search(r'\bcontrol\b', response_lower):
        label = 'Control'
    elif re.search(r'\bdementia\b|\balzheimer', response_lower):
        label = 'ProbableAD'
    elif re.search(r'\bnormal\b|\bhealthy\b', response_lower):
        label = 'Control'
    
    if label == 'Unknown':
        print(f"    [DEBUG] Could not parse label from response")
    
    return {
        'label': label,
        'mmse': int(mmse_match.group(1)) if mmse_match else -1,
        'confidence': conf_match.group(1) if conf_match else 'Unknown',
        'raw': response
    }

# ----------------- CHAT / GENERATION -----------------
def render_messages_to_text(messages: List[Dict], tokenizer) -> str:
    """Render messages to text format for model input."""
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    
    out = ""
    for m in messages:
        role = m.get("role", "")
        content = m.get("content", "")
        out += f"{role}: {content}\n"
    out += "assistant:"
    return out

def generate_from_model(text: str, tokenizer, model) -> str:
    """Generate text from model given prompt."""
    model_inputs = tokenizer(
        [text], 
        return_tensors="pt", 
        truncation=True, 
        padding=True
    ).to(next(model.parameters()).device)
    
    generated = model.generate(
        input_ids=model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        do_sample=True,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id else tokenizer.eos_token_id)
    )
    
    for in_ids, out_ids in zip(model_inputs.input_ids, generated):
        new_ids = out_ids[len(in_ids):]
        return tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return ""

# ----------------- COT PREDICTION -----------------
def predict_visit_cot(filename: str, transcript: str, tokenizer, model) -> str:
    """
    Make prediction using Chain-of-Thought prompting.
    The model is asked to reason step-by-step before giving the final answer.
    """
    cot_prompt = create_cot_prompt(filename, transcript)
    
    messages = [
        {"role": "system", "content": "You are an expert neuropsychologist diagnosing dementia from speech. Think through each aspect carefully before making your final diagnosis."},
        {"role": "user", "content": cot_prompt}
    ]
    
    text = render_messages_to_text(messages, tokenizer)
    return generate_from_model(text, tokenizer, model)

# ----------------- MAIN -----------------
def main():
    print("=" * 70)
    print("CHECKING MODEL...")
    print("=" * 70)
    hf_sanity_check(MODEL_NAME, HF_TOKEN)

    print("\n" + "=" * 70)
    print("LOADING TOKENIZER AND MODEL...")
    print("=" * 70)
    tokenizer = load_tokenizer(MODEL_NAME, hf_token=HF_TOKEN)
    
    model = load_model(
        MODEL_NAME, 
        hf_token=HF_TOKEN,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )
    
    if model:
        print(f"   Device: {next(model.parameters()).device}")

    files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".txt"))
    print(f"\n✅ Found {len(files)} transcript files")

    print("\n" + "=" * 70)
    print(f"PROCESSING TRANSCRIPTS (CHAIN-OF-THOUGHT with {MODEL_NAME})...")
    print("=" * 70)

    results = []
    for i, filename in enumerate(files, 1):
        pid = extract_patient_info(filename)
        path = Path(DATA_DIR) / filename
        print(f"\n[{i:3d}/{len(files)}] Processing {filename} (Patient ID: {pid})")
        
        try:
            with open(path, "r", encoding="utf-8") as f:
                transcript = f.read().strip()
            if not transcript:
                raise ValueError("Empty transcript")
            
            # Make prediction using COT prompting
            response = predict_visit_cot(filename, transcript, tokenizer, model)
            prediction = parse_llm_response(response)
            print(f"           → Label: {prediction['label']:12s} | MMSE: {prediction['mmse']:3d} | Confidence: {prediction['confidence']}")
            
            file_id = re.sub(r'\.txt$', '', filename)
            results.append({
                "id": file_id,
                "confidence": prediction["confidence"],
                "mmse": prediction["mmse"],
                "dx": prediction["label"]
            })
        except Exception as e:
            print(f"           ⚠️  ERROR: {e}", file=sys.stderr)
            file_id = re.sub(r'\.txt$', '', filename)
            results.append({
                "id": file_id,
                "confidence": "N/A",
                "mmse": -1,
                "dx": "Error"
            })

    print("\n" + "=" * 70)
    print("SAVING RESULTS...")
    print("=" * 70)
    df = pd.DataFrame(results, columns=["id", "confidence", "mmse", "dx"])
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Results saved to: {OUTPUT_CSV}")

    print("\n" + "=" * 70)
    print("SAMPLE PREDICTIONS (first 10 rows):")
    print("=" * 70)
    print(df.head(10).to_string(index=False))

    print(f"\n✅ Processing complete! Total: {len(results)} predictions")
    return df

if __name__ == "__main__":
    df_results = main()

c:\pytorch-cuda-workspace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CHECKING MODEL...
✅ HF CHECK OK: Qwen/Qwen3-4B (private=False)

LOADING TOKENIZER AND MODEL...
✅ Tokenizer loaded: Qwen/Qwen3-4B
[Model] Loading: Qwen/Qwen3-4B


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.98s/it]


✅ Model loaded successfully: Qwen/Qwen3-4B
   Device: cuda:0

✅ Found 552 transcript files

PROCESSING TRANSCRIPTS (CHAIN-OF-THOUGHT with Qwen/Qwen3-4B)...

[  1/552] Processing 001-0.txt (Patient ID: 001)
           → Label: ProbableAD   | MMSE:  -1 | Confidence: Unknown

[  2/552] Processing 001-2.txt (Patient ID: 001)
           → Label: ProbableAD   | MMSE:  -1 | Confidence: Unknown

[  3/552] Processing 002-0.txt (Patient ID: 002)
           → Label: ProbableAD   | MMSE:  -1 | Confidence: Unknown

[  4/552] Processing 002-1.txt (Patient ID: 002)
           → Label: ProbableAD   | MMSE:  -1 | Confidence: Unknown

[  5/552] Processing 002-2.txt (Patient ID: 002)
           → Label: ProbableAD   | MMSE:  -1 | Confidence: Unknown

[  6/552] Processing 002-3.txt (Patient ID: 002)
           → Label: ProbableAD   | MMSE:  -1 | Confidence: Since

[  7/552] Processing 003-0.txt (Patient ID: 003)
           → Label: ProbableAD   | MMSE:  -1 | Confidence: Unknown

[  8/552] Processing 005-0

In [ ]:
# eval_cot_vs_actual.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# ---------- Paths ----------
actual_path = r"E:\Thesis\2_classes.csv"
pred_path   = r"E:\Thesis\qwen3_4B_cot_binaryclass.csv"

# ---------- Load CSVs ----------
print("Loading CSVs...")
df_true = pd.read_csv(actual_path)
df_pred = pd.read_csv(pred_path)
print("Loaded actual:", actual_path)
print("Loaded predictions:", pred_path)

# ---------- Column mapping ----------
# Actual CSV columns: id, mmse, dx (or similar)
# Prediction CSV columns: id, confidence, mmse, dx

# For actual data - use mmse and dx columns
mmse_true_col = "mmse"
label_true_col = "dx"

# For prediction data - use mmse and dx columns
mmse_pred_col = "mmse"
label_pred_col = "dx"

print(f"Using (actual) mmse column: '{mmse_true_col}'  label column: '{label_true_col}'")
print(f"Using (pred)   mmse column: '{mmse_pred_col}'  label column: '{label_pred_col}'")

# ---------- Rename for consistency ----------
df_true = df_true.rename(columns={mmse_true_col: "actual_mmse", label_true_col: "actual_label"})
df_pred = df_pred.rename(columns={mmse_pred_col: "pred_mmse", label_pred_col: "pred_label"})

# ---------- Ensure 'id' exists in both ----------
if "id" not in df_true.columns or "id" not in df_pred.columns:
    raise KeyError("Both CSVs must contain an 'id' column for matching (patient id).")

# ---------- Merge on id ----------
df_merged = pd.merge(df_pred, df_true, on="id", how="inner", suffixes=("_pred", "_true"))
if df_merged.empty:
    raise ValueError("Merge resulted in 0 rows. Check that 'id' values match between files.")

print(f"Merged {len(df_merged)} rows on 'id'.")

# ---------- Normalize label strings ----------
df_merged["pred_label_norm"] = df_merged["pred_label"].astype(str).str.strip().str.lower()
df_merged["actual_label_norm"] = df_merged["actual_label"].astype(str).str.strip().str.lower()
df_merged["label_match"] = df_merged["pred_label_norm"] == df_merged["actual_label_norm"]

# ---------- Numeric MMSE ----------
df_merged["pred_mmse"] = pd.to_numeric(df_merged["pred_mmse"], errors="coerce")
df_merged["actual_mmse"] = pd.to_numeric(df_merged["actual_mmse"], errors="coerce")
df_merged["mmse_diff"] = (df_merged["pred_mmse"] - df_merged["actual_mmse"]).abs()

# ---------- Metrics ----------
label_accuracy = df_merged["label_match"].mean() * 100
mae = df_merged["mmse_diff"].mean()
corr = df_merged["pred_mmse"].corr(df_merged["actual_mmse"])

# Optional: per-class precision/recall/f1
labels = sorted(set(df_merged["actual_label_norm"].unique()) | set(df_merged["pred_label_norm"].unique()))
y_true = df_merged["actual_label_norm"]
y_pred = df_merged["pred_label_norm"]
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)

prf_df = pd.DataFrame({
    "label": labels,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support
})

# ---------- Print summary ----------
print("\n📊 Evaluation Results (Qwen3-4B Chain-of-Thought)")
print(f"Samples evaluated: {len(df_merged)}")
print(f"Label Accuracy: {label_accuracy:.2f}% ({int(df_merged['label_match'].sum())}/{len(df_merged)})")
print(f"MMSE Mean Absolute Error (MAE): {mae:.3f}")
print(f"MMSE Pearson correlation: {corr:.3f}\n")

print("Per-class precision/recall/f1:")
print(prf_df.to_string(index=False))

# ---------- Save merged CSV ----------
out_csv = r"E:\Thesis\qwen3_4B_cot_binaryclass_vs_actual_evaluation.csv"
df_merged.to_csv(out_csv, index=False)
print(f"\nSaved merged evaluation CSV: {out_csv}")

# ---------- Confusion matrix ----------
cm = confusion_matrix(df_merged["actual_label_norm"], df_merged["pred_label_norm"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

plt.figure(figsize=(8,6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Qwen3-4B COT (Binary Classification)")
confusion_path = r"E:\Thesis\qwen3_4B_cot_binaryclass_confusion_matrix.png"
plt.savefig(confusion_path, bbox_inches="tight")
plt.show()
print(f"Saved confusion matrix to: {confusion_path}")

# ---------- Scatter plot: actual vs predicted MMSE ----------
plt.figure(figsize=(6,6))
plt.scatter(df_merged["actual_mmse"], df_merged["pred_mmse"], alpha=0.7)
xmin = min(df_merged["actual_mmse"].min(), df_merged["pred_mmse"].min())
xmax = max(df_merged["actual_mmse"].max(), df_merged["pred_mmse"].max())
plt.plot([xmin, xmax], [xmin, xmax], linestyle="--", color="gray")
plt.xlabel("Actual MMSE")
plt.ylabel("Predicted MMSE")
plt.title("Predicted vs Actual MMSE - Qwen3-4B COT")
plt.grid(True)
scatter_path = r"E:\Thesis\qwen3_4B_cot_binaryclass_mmse_scatter.png"
plt.savefig(scatter_path, bbox_inches="tight")
plt.show()
print(f"Saved MMSE scatter plot to: {scatter_path}")

# ---------- Save per-class PRF ----------
prf_out = r"E:\Thesis\qwen3_4B_cot_binaryclass_per_class_prf.csv"
prf_df.to_csv(prf_out, index=False)
print(f"Saved per-class precision/recall/f1 to: {prf_out}")

# ---------- Small summary file ----------
summary = {
    "n_samples": [len(df_merged)],
    "label_accuracy_pct": [label_accuracy],
    "mmse_mae": [mae],
    "mmse_corr": [corr]
}
pd.DataFrame(summary).to_csv(r"E:\Thesis\qwen3_4B_cot_binaryclass_eval_summary.csv", index=False)
print("Saved eval summary to: E:\\Thesis\\qwen3_4B_cot_binaryclass_eval_summary.csv")